<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import pipeline
from google.colab import userdata
import pandas as pd
import numpy as np
import subprocess
import math
import json
import os

### Clone Corpus2GeoRAG Repository

In [ ]:
%%bash
rm -rf Corpus2GeoRAG
git clone https://github.com/EtzionR/Corpus2GeoRAG.git

Cloning into 'Corpus2GeoRAG'...


### Data Loading and Initial Inspection

**Purpose:** To load the raw text data, which is crucial for subsequent NER tasks, from a JSON file. This section also includes an example to visually inspect the structure and content of a single entry, ensuring the data is correctly loaded and accessible.


**Input/Output:** Input is `/content/Corpus2GeoRAG/examples/DATA.json`.

**Date:** 2026-09-23

In [ ]:
PATH = r'/content/Corpus2GeoRAG/examples/DATA.json'
PATH

'/content/Corpus2GeoRAG/examples/DATA.json'

In [ ]:
with open(PATH, 'r') as file:
    data = json.load(file)

len(data)

897

In [ ]:
[*data.keys()][:10]

['12 (song)',
 '14 May 2026 Russian strikes on Ukraine',
 '15th BRICS summit',
 '16 March 2022 Chernihiv breadline attack',
 '17 November 2024 Russian strikes on Ukraine',
 '18 March 2022 Mykolaiv military quarters attack',
 '1st European Political Community Summit',
 '2000 Meters to Andriivka',
 '2014 pro-Russian unrest in Ukraine',
 '2020s European re-armament']

In [ ]:
example = np.random.choice([*data],1)[0]


print(f'PAGE: {example}:\n\n{data[example]}')


PAGE: Ukraine's Peace Formula:

The Ukraine's Peace Formula is an initiative and diplomatic platform that Ukraine proposed to the world community to achieve a just conclusion to the Russo-Ukrainian war.


== History ==
In September 2022 Ukrainian President Volodymyr Zelenskyy addressed the UN General Assembly in a pre-recorded speech that focused on five preconditions for peace formula. There were punishment for aggression, protection of life, restoration of security and territorial integrity, security guarantees and determination to defend oneself.
On 15 November 2022 during the G20 Bali Summit Zelenskyy announced the detailed contents of the 10-point "peace formula".


== Proposals ==
The November 2022 Ukrainian proposal consists of 10 points, as reported by Ukrainian authorities and by media:

Nuclear safety, especially that of the Zaporizhzhia nuclear power plant
Food security for Asian and African countries
Energy security and restoration of Ukraine's energy infrastructure
Release

In [ ]:
TITLE = '=='
MIN_LENGTH = 5

paragraphs = data[example].split('\n')
paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]

print(f'{len(paragraphs_processed)} Paragraphs Extracted from {len(paragraphs)}\n{round(len(paragraphs_processed)/len(paragraphs)*100,1)}%')

32 Paragraphs Extracted from 55
58.2%


### Text Preprocessing: Paragraph Filtering

**Purpose:** To perform an initial cleaning of the raw text by filtering out structural elements and short, uninformative lines. This step aims to prepare cleaner text segments for downstream NER processing by removing elements such as section titles and empty or minimal lines.

**Date:** 2026-09-24

In [ ]:
TITLE = '=='
MIN_LENGTH = 5

data_processed = {}

total_paragraphs = 0
total_processed = 0

for key, text in data.items():
  paragraphs = text.split('\n')
  paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]
  data_processed[key] = paragraphs_processed

  total_paragraphs += len(paragraphs)
  total_processed += len(paragraphs_processed)

print(f'{total_processed} paragraphs extracted from {total_paragraphs}\n {round(total_processed/total_paragraphs*100,1)}%')

42822 paragraphs extracted from 88029
 48.6%


In [ ]:
example = np.random.choice([*data_processed],1)[0]

print(f'PAGE: {example}:\n\n{data_processed[example]}')

PAGE: Ukrainian cultural heritage during the 2022 Russian invasion:

["Russia began the full-scale invasion of Ukraine on 24 February 2022, in a major escalation of the Russo-Ukrainian War that began in 2014. It is the largest military attack in Europe since World War II. During the fighting, many pieces of Ukrainian cultural heritage were either destroyed, damaged, or put at risk due to the widespread destruction across the country. This deliberate destruction and looting of over 500 Ukrainian cultural heritage sites is considered as a war crime and has been described by Ukraine's Minister of Culture as cultural genocide.", 'Cultural property enjoys special protection under international humanitarian law. The first protocol of the Geneva Convention and the Hague Convention for the Protection of Cultural Property in the Event of Armed Conflict (both binding on the Ukraine and Russia settlements) prohibit states parties from using historic monuments to support their military effort or m

In [ ]:
SAVE_PATH = '/content/Corpus2GeoRAG/ApplyNER/data_initial_cleaning.json'

with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(data_processed, f, ensure_ascii=False, indent=2)

print(f'Saved to: {SAVE_PATH}')

Saved to: /content/Corpus2GeoRAG/ApplyNER/data_initial_cleaning.json


## Initial NER application

**Purpose:** To perform an initial NER application with base models and analyze the results

**Date:** 2026-09-24

### example

#### DistilBERT

In [ ]:
MODEL_DistilBERT = "dslim/distilbert-NER"
MODEL_DistilBERT

ner_DistilBERT = pipeline("ner",
               model=MODEL_DistilBERT,
               aggregation_strategy='average')

ner_DistilBERT

config.json:   0%|          | 0.00/926 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/669k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

TokenClassificationPipeline: {'model': 'DistilBertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [ ]:
output_DistilBERT = ner_DistilBERT(data_processed[example])
output_DistilBERT

[[{'entity_group': 'LOC',
   'score': np.float32(0.9989077),
   'word': 'Russia',
   'start': 0,
   'end': 6},
  {'entity_group': 'LOC',
   'score': np.float32(0.9986571),
   'word': 'Ukraine',
   'start': 40,
   'end': 47},
  {'entity_group': 'MISC',
   'score': np.float32(0.99675876),
   'word': 'Russo',
   'start': 98,
   'end': 103},
  {'entity_group': 'MISC',
   'score': np.float32(0.9954928),
   'word': '-',
   'start': 103,
   'end': 104},
  {'entity_group': 'MISC',
   'score': np.float32(0.9332204),
   'word': 'Ukrainian War',
   'start': 104,
   'end': 117},
  {'entity_group': 'LOC',
   'score': np.float32(0.9977196),
   'word': 'Europe',
   'start': 175,
   'end': 181},
  {'entity_group': 'MISC',
   'score': np.float32(0.983453),
   'word': 'World War II',
   'start': 188,
   'end': 200},
  {'entity_group': 'MISC',
   'score': np.float32(0.99722725),
   'word': 'Ukrainian',
   'start': 238,
   'end': 247},
  {'entity_group': 'MISC',
   'score': np.float32(0.99638414),
   'wor

In [ ]:
df_DistilBERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_DistilBERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,LOC,0.998908,Russia,0,6
1,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,LOC,0.998657,Ukraine,40,47
2,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,MISC,0.996759,Russo,98,103
3,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,MISC,0.995493,-,103,104
4,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,MISC,0.933220,Ukrainian War,104,117
...,...,...,...,...,...,...,...,...
394,Ukrainian cultural heritage during the 2022 Ru...,50,"Egan, Lauren (2022-05-30). ""Ukrainians see the...",LOC,0.996311,Russia,73,79
395,Ukrainian cultural heritage during the 2022 Ru...,50,"Egan, Lauren (2022-05-30). ""Ukrainians see the...",ORG,0.859922,NBC News.,101,110
396,Ukrainian cultural heritage during the 2022 Ru...,51,"""Map of cultural losses"". Ukrainian Cultural F...",ORG,0.992799,Ukrainian Cultural Foundation,26,55
397,Ukrainian cultural heritage during the 2022 Ru...,52,List of damaged objects by Ukrainian Ministry ...,MISC,0.698304,Ukrainian,27,36


#### BERT

In [ ]:
MODEL_BERT = "dslim/bert-base-NER"
MODEL_BERT

ner_BERT = pipeline("ner",
               model=MODEL_BERT,
               aggregation_strategy='average')

ner_BERT

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

TokenClassificationPipeline: {'model': 'BertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [ ]:
output_BERT = ner_BERT(data_processed[example])
output_BERT

[[{'entity_group': 'LOC',
   'score': np.float32(0.99982953),
   'word': 'Russia',
   'start': 0,
   'end': 6},
  {'entity_group': 'LOC',
   'score': np.float32(0.9998275),
   'word': 'Ukraine',
   'start': 40,
   'end': 47},
  {'entity_group': 'MISC',
   'score': np.float32(0.98833877),
   'word': 'Russo - Ukrainian War',
   'start': 98,
   'end': 117},
  {'entity_group': 'LOC',
   'score': np.float32(0.9997217),
   'word': 'Europe',
   'start': 175,
   'end': 181},
  {'entity_group': 'MISC',
   'score': np.float32(0.9977885),
   'word': 'World War II',
   'start': 188,
   'end': 200},
  {'entity_group': 'MISC',
   'score': np.float32(0.9996915),
   'word': 'Ukrainian',
   'start': 238,
   'end': 247},
  {'entity_group': 'MISC',
   'score': np.float32(0.99970156),
   'word': 'Ukrainian',
   'start': 419,
   'end': 428},
  {'entity_group': 'LOC',
   'score': np.float32(0.99980074),
   'word': 'Ukraine',
   'start': 508,
   'end': 515}],
 [{'entity_group': 'MISC',
   'score': np.float32

In [ ]:
df_BERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_BERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,LOC,0.998908,Russia,0,6
1,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,LOC,0.998657,Ukraine,40,47
2,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,MISC,0.996759,Russo,98,103
3,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,MISC,0.995493,-,103,104
4,Ukrainian cultural heritage during the 2022 Ru...,0,Russia began the full-scale invasion of Ukrain...,MISC,0.933220,Ukrainian War,104,117
...,...,...,...,...,...,...,...,...
394,Ukrainian cultural heritage during the 2022 Ru...,50,"Egan, Lauren (2022-05-30). ""Ukrainians see the...",LOC,0.996311,Russia,73,79
395,Ukrainian cultural heritage during the 2022 Ru...,50,"Egan, Lauren (2022-05-30). ""Ukrainians see the...",ORG,0.859922,NBC News.,101,110
396,Ukrainian cultural heritage during the 2022 Ru...,51,"""Map of cultural losses"". Ukrainian Cultural F...",ORG,0.992799,Ukrainian Cultural Foundation,26,55
397,Ukrainian cultural heritage during the 2022 Ru...,52,List of damaged objects by Ukrainian Ministry ...,MISC,0.698304,Ukrainian,27,36


### Model testing and evaluation (done on 10% of the data)

#### subset creation

In [ ]:
all_pages = list(data_processed.keys())

if not all_pages:
    raise ValueError("data_processed contains no pages.")

sample_size = max(1, math.ceil(len(all_pages) * 0.10))

evaluation_pages = (
    pd.Series(all_pages)
    .sample(n=sample_size)
    .tolist()
)

evaluation_data = {
    page: data_processed[page]
    for page in evaluation_pages
}

print(f"Sampled {sample_size} of {len(all_pages)} pages.")

#### DistilBERT

In [ ]:

rows = []

for page, paragraphs in evaluation_data.items():
    for paragraph_id, paragraph in enumerate(paragraphs):
        entities = ner_DistilBERT(paragraph)

        for entity in entities:
            rows.append({
                "entity_group": entity["entity_group"],
                "score": entity["score"],
                "word": entity["word"],
                "start": entity["start"],
                "end": entity["end"],
                "Source": page,
                "Index": paragraph_id,
                "Text": paragraph
            })

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_DistilBERT_10pct = pd.DataFrame(rows, columns=columns)

output_dir = os.path.join(
    "NER_output",
    "model_testing_and_evaluation"
)
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "post_NER_DistilBERT_10pct.csv"
)

df_DistilBERT_10pct.to_csv(output_path, index=False)

print(f"Processed {len(evaluation_data)} sampled pages.")
print(f"Saved {len(df_DistilBERT_10pct)} entity rows to: {output_path}")

df_DistilBERT_10pct.head()

#### BERT

In [ ]:
rows = []

for page, paragraphs in evaluation_data.items():
    for paragraph_id, paragraph in enumerate(paragraphs):
        entities = ner_BERT(paragraph)

        for entity in entities:
            rows.append({
                "entity_group": entity["entity_group"],
                "score": entity["score"],
                "word": entity["word"],
                "start": entity["start"],
                "end": entity["end"],
                "Source": page,
                "Index": paragraph_id,
                "Text": paragraph
            })

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_BERT_10pct = pd.DataFrame(rows, columns=columns)

output_dir = os.path.join(
    "NER_output",
    "model_testing_and_evaluation"
)
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "post_NER_DistilBERT_10pct.csv"
)

df_BERT_10pct.to_csv(output_path, index=False)

print(f"Processed {len(evaluation_data)} sampled pages.")
print(f"Saved {len(df_DistilBERT_10pct)} entity rows to: {output_path}")

df_BERT_10pct.head()

### Full Data

on chosen model (currently DistilBert pending models evaluations)
date: 05.10.26

In [ ]:
rows = []

for page, paragraphs in data_processed.items():
    for paragraph_id, paragraph in enumerate(paragraphs):
        entities = ner_DistilBERT(paragraph)

        for entity in entities:
            rows.append({
                "entity_group": entity["entity_group"],
                "score": entity["score"],
                "word": entity["word"],
                "start": entity["start"],
                "end": entity["end"],
                "Source": page,
                "Index": paragraph_id,
                "Text": paragraph
            })

columns = [
    "entity_group",
    "score",
    "word",
    "start",
    "end",
    "Source",
    "Index",
    "Text"
]

df_DistilBERT = pd.DataFrame(rows, columns=columns)

output_dir = "NER_output"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "post_NER.csv")
df_DistilBERT.to_csv(output_path, index=False)

print(f"Processed {len(data_processed)} pages.")
print(f"Saved {len(df_DistilBERT)} entity rows to {output_path}")

## Commit to repo

In [ ]:
REPO_PATH = "/content/Corpus2GeoRAG"
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

os.chdir(REPO_PATH)

def run_git(args, *, env=None, capture_output=False):
    """Run Git and raise an error that includes Git's actual output."""
    result = subprocess.run(
        ["git", *args],
        env=env,
        text=True,
        capture_output=capture_output
    )

    if capture_output:
        if result.stdout:
            print(result.stdout, end="")
        if result.stderr:
            print(result.stderr, end="")

    if result.returncode != 0:
        if capture_output:
            details = (result.stderr or result.stdout or "").strip()
        else:
            details = f"Git exited with status {result.returncode}."
        raise RuntimeError(f"git {' '.join(args)} failed:\n{details}")

    return result


# --------------------------------------------------
# Git configuration
# --------------------------------------------------

run_git(["config", "user.name", GITHUB_USER])
run_git(["config", "user.email", GITHUB_EMAIL])
run_git([
    "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
])

branch = run_git(
    ["branch", "--show-current"],
    capture_output=True
).stdout.strip()

if not branch:
    raise RuntimeError("Could not determine the current Git branch.")

# --------------------------------------------------
# GitHub authentication
# --------------------------------------------------

github_token = userdata.get("GITHUB_TOKEN")

if not github_token:
    raise ValueError(
        "GITHUB_TOKEN is missing from Colab Secrets. "
        "Add it in the Colab Secrets panel and enable notebook access."
    )

askpass_path = "/tmp/github_askpass.sh"

with open(askpass_path, "w") as f:
    f.write(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "$GITHUB_USER" ;;\n'
        '  *Password*) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        "esac\n"
    )

os.chmod(askpass_path, 0o700)

git_env = os.environ.copy()
git_env["GITHUB_USER"] = GITHUB_USER
git_env["GITHUB_TOKEN"] = github_token
git_env["GIT_ASKPASS"] = askpass_path
git_env["GIT_TERMINAL_PROMPT"] = "0"

try:
    # --------------------------------------------------
    # Prevent accidental nested repository
    # --------------------------------------------------

    # Keep this rule only if /content/Corpus2GeoRAG is an accidental
    # nested folder inside the repository. It does not ignore the repo itself.
    ignore_entry = "/Corpus2GeoRAG/"
    gitignore_path = os.path.join(REPO_PATH, ".gitignore")

    existing_ignore = ""
    if os.path.exists(gitignore_path):
        with open(gitignore_path, "r") as f:
            existing_ignore = f.read()

    if ignore_entry not in existing_ignore.splitlines():
        with open(gitignore_path, "a") as f:
            if existing_ignore and not existing_ignore.endswith("\n"):
                f.write("\n")
            f.write(ignore_entry + "\n")

    # If this nested path was already tracked, stop tracking it.
    nested_repo = run_git(
        ["ls-files", "--stage", "--", "Corpus2GeoRAG"],
        capture_output=True
    ).stdout.strip()

    if nested_repo:
        run_git(["rm", "--cached", "-r", "-f", "--", "Corpus2GeoRAG"])
        print("Removed the nested Corpus2GeoRAG path from Git tracking.")

    # --------------------------------------------------
    # Stage repository changes, except local notebook copies
    # --------------------------------------------------

    run_git(["add", "-A"])

    staged_files = run_git(
        ["diff", "--cached", "--name-only"],
        capture_output=True
    ).stdout.splitlines()

    for notebook in staged_files:
        if notebook.endswith(".ipynb"):
            run_git(["restore", "--staged", "--", notebook])

    # --------------------------------------------------
    # Commit staged changes, if any
    # --------------------------------------------------

    staged_diff = subprocess.run(
        ["git", "diff", "--cached", "--quiet"],
        text=True
    )

    if staged_diff.returncode == 1:
        run_git(["commit", "-m", COMMIT_MESSAGE])
        print("Repository changes committed.")
    elif staged_diff.returncode == 0:
        print("No new repository changes to commit.")
    else:
        raise RuntimeError("Could not check the staged changes.")

    # --------------------------------------------------
    # Restore tracked notebooks to the committed versions
    # --------------------------------------------------

    tracked_files = run_git(
        ["ls-tree", "-r", "--name-only", "HEAD"],
        capture_output=True
    ).stdout.splitlines()

    for notebook in tracked_files:
        if notebook.endswith(".ipynb"):
            run_git(["restore", "--worktree", "--", notebook])

    # --------------------------------------------------
    # Pull/rebase GitHub changes
    # --------------------------------------------------

    pull = subprocess.run(
        ["git", "pull", "--rebase", "origin", branch],
        env=git_env,
        text=True,
        capture_output=True
    )

    if pull.stdout:
        print(pull.stdout, end="")
    if pull.stderr:
        print(pull.stderr, end="")

    if pull.returncode != 0:
        # The known conflict from the earlier run: both sides added
        # .gitignore, and both contained the same single rule.
        conflicted = run_git(
            ["diff", "--name-only", "--diff-filter=U"],
            capture_output=True
        ).stdout.splitlines()

        if conflicted == [".gitignore"]:
            ours = run_git(
                ["show", ":2:.gitignore"],
                capture_output=True
            ).stdout
            theirs = run_git(
                ["show", ":3:.gitignore"],
                capture_output=True
            ).stdout

            if ours.strip() == theirs.strip():
                with open(".gitignore", "w") as f:
                    f.write(ours)
                    if ours and not ours.endswith("\n"):
                        f.write("\n")

                run_git(["add", "--", ".gitignore"])
                run_git(
                    ["-c", "core.editor=true", "rebase", "--continue"]
                )
            else:
                raise RuntimeError(
                    "The pull stopped on a .gitignore conflict, and the two "
                    "versions differ. The rebase is paused; inspect the "
                    "conflict before continuing."
                )
        else:
            raise RuntimeError(
                "The pull failed. The rebase may be paused on a conflict. "
                "Check `git status` before running this cell again."
            )

    # --------------------------------------------------
    # Push the synchronized branch
    # --------------------------------------------------

    run_git(["push", "origin", branch], env=git_env)
    print(f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} on branch '{branch}'.")

finally:
    if os.path.exists(askpass_path):
        os.remove(askpass_path)

main
No new repository changes to commit.
.gitignore
ApplyNER/NER.ipynb
ApplyNER/data_initial_cleaning.json
Graph_n_Geocoding/Graph_and_Geocoding.ipynb
LICENSE
README.md
examples/DATA.json
examples/entities_table.csv
Already up to date.
From https://github.com/idoschw3/Corpus2GeoRAG
 * branch            main       -> FETCH_HEAD
Successfully pushed to idoschw3/Corpus2GeoRAG on branch 'main'.
